# 다른 데이터셋 그래프200 실제 답변·채점 비교

CPU 런타임에서 Run all 하십시오. 파일 업로드 없이 기존 Drive의 고정 비공개 입력을 읽습니다.

이번 고정 대상: MemConflict 552 changed questions and Memora 167 changed questions; verified snapshot, not full benchmark.

최대 세 묶음이 동시에 실행됩니다. 한 묶음의 실패는 다른 묶음을 막지 않습니다. 기존 KnowMe 401회는 다시 호출하지 않습니다. 전체 추가 평가 예산 상한 $30이며 이번 파일의 고정 배분 합계는 $12.0입니다.

30초 이내 자식 진행 출력과 20초 부모 상태/Drive 체크포인트를 남깁니다. 실패·미준비 벤치는 성능 완료로 집계하지 않습니다. OPENROUTER_API_KEY가 필요하며 BEAM 원 판정 경로를 포함할 때는 OPENAI_API_KEY도 필요합니다. 키 값은 출력하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
archive_name = 'MUSTER_GRAPH_OTHER_QUALITY_STORE26_PRIVATE_v1.zip'
archive_sha = 'd1134c87736629c82fa9fa6f2bc4b301bd0a2ba13961b67d29e4a0f5cddf3345'
candidates = [Path('/content/drive/MyDrive/MUSTER_graph_notebooks_20261116/private_inputs') / archive_name,
              Path('/content/drive/MyDrive/MUSTER_graph_quality_others_20261116/inputs') / archive_name]
archive = next((p for p in candidates if p.is_file()), None)
if archive is None:
    raise RuntimeError('사전 배치된 비공개 입력을 Drive에서 찾지 못했습니다. 입력을 임의 업로드해 대체하지 않습니다.')
if hashlib.sha256(archive.read_bytes()).hexdigest() != archive_sha:
    raise RuntimeError('비공개 입력 ZIP SHA가 다릅니다. 새 요청을 시작하지 않았습니다.')
root = Path('/content/muster_other_quality_20261116') / archive_sha[:16]
package = root / 'package'
package.mkdir(parents=True, exist_ok=True)
if (package / 'BATCH.json').exists():
    if hashlib.sha256((package / 'BATCH.json').read_bytes()).hexdigest() != 'ddfe1b711c023b0bf94faee588c7af3c37e760f954783ad4cf9facb2cc2598e2':
        raise RuntimeError('기존 실행 폴더의 고정 입력이 다릅니다. 실행 중 파일을 덮지 않습니다.')
    prior = json.loads((package / 'BATCH.json').read_text())
    for rel, digest in prior['files'].items():
        if hashlib.sha256((package / rel).read_bytes()).hexdigest() != digest:
            raise RuntimeError('기존 파일 SHA가 다릅니다. 새 요청 없이 중단합니다.')
    print('동일한 고정 입력을 재사용합니다. 실행 중 소스·입력을 덮지 않습니다.', flush=True)
else:
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            rel = Path(entry.filename)
            if rel.is_absolute() or '..' in rel.parts or ((entry.external_attr >> 16) & 0o170000) == 0o120000:
                raise RuntimeError('입력 ZIP 경로가 안전하지 않습니다.')
            target = package / rel
            if entry.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(z.read(entry))
print('고정 입력 확인 완료:', ['MemConflict 552 changed questions and Memora 167 changed questions; verified snapshot, not full benchmark'], flush=True)


In [ ]:
venv = root / 'venv'
if not (venv / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(venv)], check=True)
python = str(venv / 'bin/python')
command = [sys.executable, '-m', 'pip', '--python', python, 'install', '--quiet', '-r', str(package / 'PACKAGE_REQUIREMENTS.txt')]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError('CPU 실행 환경 설치에 실패했습니다. 위 실제 출력을 보존합니다.')
print('CPU 환경 준비 완료. GPU 설치·모델 적재 없음.', flush=True)


In [ ]:
from google.colab import userdata
env = dict(os.environ)
for target, names in {'OPENROUTER_API_KEY': ['OPENROUTER_API_KEY', 'OPENROUTER_KEY'],
                      'OPENAI_API_KEY': ['OPENAI_API_KEY']}.items():
    if not env.get(target):
        for name in names:
            try:
                value = userdata.get(name)
            except Exception:
                value = None
            if value:
                env[target] = value
                break
out = Path('/content/drive/MyDrive/MUSTER_graph_quality_others_20261116/results') / archive_sha[:16]
local = root / 'work'
common = [python, '-u', str(package / 'others_run.py'), '--package', str(package),
          '--manifest-sha256', 'ddfe1b711c023b0bf94faee588c7af3c37e760f954783ad4cf9facb2cc2598e2', '--out', str(out), '--local', str(local), '--python', python]
for mode in ['plan', 'run']:
    print('추가 평가:', mode, '| Drive 결과:', out, flush=True)
    with subprocess.Popen(common + ['--mode', mode], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=env) as p:
        try:
            for line in p.stdout:
                print(line, end='', flush=True)
            code = p.wait()
        except BaseException:
            p.send_signal(2)
            p.wait(timeout=25)
            raise
        if code:
            raise RuntimeError('일부 묶음이 실패·차단됐습니다. 끝난 묶음은 Drive에 보존됐고 전체 완료로 집계하지 않습니다.')
print('이번 고정 묶음 완료. 벤치별 SUMMARY와 원장 backup을 Drive에 보존했습니다.', flush=True)
